<center style="padding:1rem 0;">
    <h1 style="font-size: 4rem;">IA - Deep Learning</h1>
    <h2 style="font-size: 2rem;">Livrable 2 - Construction d'un premier réseau de neurones</h2>
    <h5 style="font-size: 1rem;"><i>Thomas VINET, Hugo HELM, Alban GODIER</i></h5>
</center>

<img src="assets/cesi.png" style="position:absolute;right:2rem;top:4.5rem;width:10rem;background:#fee237;"/>

In [14]:
from __future__ import annotations
from typing import Optional, Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.neural_network import NeuralNetwork, DrawRealTimeLoss, EarlyStopping, Layer, Evaluation
from lib.neural_network.grid_search import GridSearch

## Démonstration mathématiques

Lors de la descente de gradient dans note modèle, nous cherchons à ajuster les poids et les biais de chaque couche du réseau de neurone. Pour ce faire, nous partons de la sortie du réseau pour chercher à optimiser le résultat de la fonction de coût. 

### Introduction des variables

On considère un réseau de neurones entièrement connecté, composé de couches indexées par $i \in \{0, \dots, L\}$ où :
- $i = 0$ : couche de sortie
- $i = L$ : couche d’entrée

On pose :
- $y$ : la valeur attendue pour la prédiction
- $\hat y$ : la prédiction du modèle
- $\mathcal L$ la fonction de coût du modèle, définie en fonction de $y$ et $\hat y$
- $A_i$ la fonction d'activation de la couche $i$, appliquée composant par composant
- $z_{i}$ la matrice des sorties des fonctions d'agrégation des neurones de la couche $i$
- $x_i$ la matrice des entrées pour les neurones de la couche $i$
- $w_{i}$ la matrice des poids liés aux entrées $x_i$ pour les neurone de la couche $i$
- $b_{i}$ la matrice des biais des neurones de la couche $i$

Les dimensions des matrices sont les suivantes :
- $x_L$ : $\text{variables entrée} \times 1$ , la matrice a une ligne par variable d'entrée (sortie de la couche $i+1$) et 1 colonne.
- $x_i$ : $\text{Neurones}_{i+1} \times 1$, la matrice a une ligne par neurone de la couche précédente ($i+1$) et 1 colonne.
- $w_i$ : $\text{Neurones}_i \times \text{Neurones}_{i+1}$ , la matrice a une ligne par neurone de la couche $i$ et une colonne par neurone de la couche $i+1$
- $b_i$ : $\text{Neurones}_i \times 1$ , la matrice a une ligne par neurone de la couche $i$ et 1 colonne.
- $z_i$ : $\text{Neurones}_i \times 1$, la matrice a une ligne par neurone de la couche $i$ et 1 colonne.

Pour chaque couche du réseau :
$$
\begin{cases}
z_i = w_ix_i + b_i \\
x_{i-1} = A_i(z_i)
\end{cases}
$$

On note également la prédiction du modèle et la fonction de coût :
- $\hat y = A_0(z_0)$
- $\mathcal L = \mathcal L(y, \hat y)$

Dans le cas de la descente de gradient, on cherche à connaitre :
$$\frac{\partial \mathcal L}{\partial w_i} \quad et \quad \frac{\partial \mathcal L}{\partial b_i}$$
Pour cela, on pose la variable :
$$\delta_i = \frac{\partial \mathcal L}{\partial z_i}$$

Nous cherchons donc à définir tous les gradients des poids et des biais en fonction de ces valeurs.
Afin d'avoir une formule plus dynamique pour le modèle, nous voulons obtenir des expressions récurrentes, en afin d'obtenir le gradient d'une couche $i$ en fonction de la couche précédente ($i-1$)

### Exemple pour la couche 0

Afin de démontrer notre relation récursive, on commence par définir le gradient de la dernière couche (couche de sortie).
On a :
- $\hat y = A_0(z_0)$, $A_0$ étant la fonction d'activation de la dernière couche
- $z_0=w_0x_0+b_0$

La matrice $\hat y$ est de forme $\text{Neurones}_0 \times 1$ (matrice colonne dont le nombre de ligne correspond au nombre de sortie de la couche de sortie du modèle)

On cherche : $\frac{\partial \mathcal L}{\partial w_0}$ et $\frac{\partial \mathcal L}{\partial b_0}$
Pour cela on cherche $\delta_0$ afin de les obtenir par composition.


On pose :
$$
\begin{align}
\delta_0 &= \frac{\partial \mathcal L}{\partial \hat y} \odot \frac{\partial \hat y}{\partial z_0} \\
&= \frac{\partial \mathcal L}{\partial \hat y} \odot A'_0(z_0)
\end{align}
$$
Donc :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial w_0}
	& =
	\delta_0 \cdot \left ( \frac{\partial z_0}{\partial w_0} \right )^\intercal \\
	& =
	\delta_0 \cdot x_0^\intercal
\end{align}
$$
Comme $\delta_0$ est une matrice colonne ($\text{Neurones}_0 \times 1$) et $x_0$ ($\text{Neurones}_1 \times 1$) l'est aussi, on multiplie $\delta_0$ par par la transposée de $x_0$. de cette manière, on obtient une matrice de forme $\text{Neurones}_0 \times \text{Neurones}_1$ que l'on peut donc soustraire à $w_0$. 
Et :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial b_0}
	& =
	\delta_0 \cdot \frac{\partial z_0}{\partial b_0} \\
	& =
	\delta_0
\end{align}
$$

### Exemple pour la couche 1

On définit ensuite le gradient de notre première couche cachée (couche 1).
On a :
- $z_0=w_0 x_0 + b_0$
- $x_0=A_1(z_1)$
- $z_1=w_1 x_1 + b_1$

On cherche : $\frac{\partial \mathcal L}{\partial w_1}$ et $\frac{\partial \mathcal L}{\partial b_1}$
Pour cela on cherche $\delta_1$, comme pour la dernière couche.
On pose :
$$
\begin{align}
	\delta_1
	& =
	\left ( \left ( 
	 \frac
	  {\partial z_0}
	  {\partial x_0}
	 \right )^\intercal
	\cdot
	\frac
	 {\partial \mathcal L}
	 {\partial \hat y} 
	\cdot 
	\frac
	 {\partial \hat y}
	 {\partial z_0}
	\right )
	\odot
	\frac
	 {\partial x_0}
	 {\partial z_1} \\
	& =
	(	w_0^\intercal
	\cdot
	\delta_0)
	\odot
	A_1'(z_1)
\end{align}
$$
$w_0$ est de dimensions $\text{Neurones}_0 \times \text{Neurones}_1$ et $\delta_0$ est de dimensions $\text{Neurones}_0 \times 1$ nous calculons la transposée de $w_0$ avant de la multiplier avec $\delta_0$.
On obtient alors une matrice de dimensions $\text{Neurones}_1 \times 1$ qui peut être multipliée par élément avec $A_1'(z_1)$ qui est aussi de dimensions $\text{Neurones}_1 \times 1$.
Donc :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial w_1}
	& =
	\delta_1 \cdot \left ( \frac{\partial z_1}{\partial w_1} \right )^\intercal \\
	& =
	\delta_1 \cdot x_1^\intercal
\end{align}
$$
Et :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial b_1}
	& =
	\delta_1 \cdot \frac{\partial z_1}{\partial b_1}  \\
	& =
	\delta_1
\end{align}
$$

### Définition de la relation récursive

En partant de l'expression de la couche 1 :
$$
\begin{align}
	\delta_1
	& =
	(
	w_0^\intercal
	\cdot
	\delta_0)
	\odot
	A_1'(z_1)
\end{align}
$$
On a les dimensions :
- $\delta_i$ : $\text{Neurones}_i \times 1$
- $A_i'(z_i)$ : $\text{Neurones}_i \times 1$

En prenant se basant sur les exemples précédents, on peut définir la relation suivante :

$$
\begin{align}
	\delta_i
	& =
	(w_{i-1}^\intercal
	\cdot
	\delta_{i-1})
	\odot
	A_i'(z_i)
\end{align}
$$
Pour les poids :
$$
\frac{\partial \mathcal L}{\partial w_i}
=
\delta_i \times x_i^\intercal
$$
Qui est donc de dimensions : $\text{Neurones}_i \times \text{Neurones}_{i+1}$

Pour les biais :
$$
\frac{\partial \mathcal L}{\partial b_i} = \delta_i
$$
Qui est donc de dimensions : $\text{Neurones}_i \times 1$

#### Cas des batches

Dans le cas des batches, les dimensions de certaines matrices changent :
- $x_L$ : $\text{variables entrée} \times \text{batches}$ , la matrice a une ligne par variable d'entrée et une colonne par batch.
- $x_i$ : $\text{Neurones}_{i+1} \times \text{batches}$, la matrice a une ligne par neurone de la couche précédente ($i+1$) et une colonne par batch.
- $b_i$ : $\text{Neurones}_i \times \text{batches}$ , la matrice a une ligne par neurone de la couche $i$ et une colonne par batch.
- $z_i$ : $\text{Neurones}_i \times \text{batches}$, la matrice a une ligne par neurone de la couche $i$ et une colonne par batch.

Cela a principalement un effet sur les dimensions de la sortie :
- $\hat y$ : $1 \times \text{batches}$

Ce qui change donc la formule du gradient des biais, comme $\delta_i$ est désormais de dimension $\text{Neurones}_i \times \text{batches}$ (dépendant de $\hat y$), cette formule devient donc :
$$\frac{\partial \mathcal L}{\partial b_i} = \text{Moy}_{\text{col}}(\delta_i)$$
Où $\text{Moy}_{\text{col}}(\delta_i)$ est la moyenne de $\delta_i$ calculée sur chaque ligne, ce qui renvoie une matrice de dimension $\text{Neurones}_i \times 1$.

# Doc TEST

## 1. Chargement des données

Chargement des données d'entraînement et de validation à partir des fichiers CSV.

In [2]:
# Chargement des données
df_train = pd.read_csv('dataset/dataset_train.csv')
df_validation = pd.read_csv('dataset/dataset_validation.csv')

print(f"Données d'entraînement: {df_train.shape}")
print(f"Données de validation: {df_validation.shape}")
print(f"\nPremières lignes des données d'entraînement:")
print(df_train.head())
print(f"\nInformations sur les données:")
print(df_train.info())

Données d'entraînement: (59582, 17)
Données de validation: (22384, 17)

Premières lignes des données d'entraînement:
   Diabetes_binary  HighBP  HighChol  CholCheck      BMI  Smoker  Stroke  \
0              0.0     0.0       0.0        1.0  0.31250     0.0     0.0   
1              0.0     1.0       0.0        1.0  0.34375     1.0     0.0   
2              1.0     1.0       1.0        1.0  0.75000     0.0     0.0   
3              1.0     1.0       0.0        1.0  0.53125     0.0     0.0   
4              0.0     1.0       1.0        1.0  0.43750     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  \
0                   0.0           1.0     1.0      1.0                0.0   
1                   0.0           1.0     0.0      1.0                0.0   
2                   1.0           0.0     1.0      1.0                0.0   
3                   0.0           0.0     0.0      1.0                0.0   
4                   0.0           0.0    

In [3]:
# Séparation des features et de la cible
# La colonne cible est 'Diabetes_binary'
target_column = 'Diabetes_binary'

X_train = df_train.drop(columns=[target_column]).values.astype(np.float32)
y_train = df_train[target_column].values.astype(np.int32)

X_validation = df_validation.drop(columns=[target_column]).values.astype(np.float32)
y_validation = df_validation[target_column].values.astype(np.int32)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_validation shape: {X_validation.shape}, y_validation shape: {y_validation.shape}")
print(f"\nDistribution de y_train: {np.bincount(y_train)}")
print(f"Distribution de y_validation: {np.bincount(y_validation)}")

X_train shape: (59582, 16), y_train shape: (59582,)
X_validation shape: (22384, 16), y_validation shape: (22384,)

Distribution de y_train: [29791 29791]
Distribution de y_validation: [19074  3310]


## 2. Choix de l'Architecture du Réseau

### Variation de l'architecture avec différentes configurations
- Couches et nombre de neurones
- Fonctions d'activation : Sigmoid, ReLU, Tanh, Softmax
- Fonctions de perte : MSE, MAE, BCE, CCE
- Dropouts pour la régularisation

In [4]:
# Imports des fonctions d'activation et de perte disponibles
from lib.neural_network.activation.sigmoid import Sigmoid
from lib.neural_network.activation.relu import Relu
from lib.neural_network.activation.tanh import Tanh

from lib.neural_network.loss import (
    BinaryCrossEntropy,
    CategoricalCrossEntropy, 
    MeanSquaredError,
    MeanAbsoluteError
)

# Dictionnaires pour faciliter l'accès aux activations et loss functions
activations_dict = {
    'sigmoid': Sigmoid,
    'relu': Relu,
    'tanh': Tanh,
}

loss_dict = {
    'bce': BinaryCrossEntropy,
    'cce': CategoricalCrossEntropy,
    'mse': MeanSquaredError,
    'mae': MeanAbsoluteError
}


print("Activations disponibles:", list(activations_dict.keys()))
print("Loss functions disponibles:", list(loss_dict.keys()))

Activations disponibles: ['sigmoid', 'relu', 'tanh']
Loss functions disponibles: ['bce', 'cce', 'mse', 'mae']


In [5]:
# Définition des configurations d'architecture à tester
architectures = [
    {
        'name': 'Simple (64-32)',
        'layers': [64, 32],
        'activation': 'relu',
        'dropout': [0.0, 0.0],
    },
    {
        'name': 'Simple avec Dropout (64-32)',
        'layers': [64, 32],
        'activation': 'relu',
        'dropout': [0.2, 0.2],
    },
    {
        'name': 'Moyenne (128-64-32)',
        'layers': [128, 64, 32],
        'activation': 'relu',
        'dropout': [0.0, 0.0, 0.0],
    },
    {
        'name': 'Moyenne avec Dropout (128-64-32)',
        'layers': [128, 64, 32],
        'activation': 'relu',
        'dropout': [0.2, 0.2, 0.1],
    },
    {
        'name': 'Profonde (256-128-64-32)',
        'layers': [256, 128, 64, 32],
        'activation': 'relu',
        'dropout': [0.2, 0.2, 0.1, 0.1],
    },
    {
        'name': 'Mixed Activations (128-64)',
        'layers': [128, 64],
        'activation': 'tanh',
        'dropout': [0.1, 0.1],
    },
]

# Loss functions à tester
loss_functions_to_test = ['bce', 'mse']

print(f"Configurations d'architectures à tester: {len(architectures)}")
for arch in architectures:
    print(f"  - {arch['name']}: {arch['layers']}, activation={arch['activation']}")

Configurations d'architectures à tester: 6
  - Simple (64-32): [64, 32], activation=relu
  - Simple avec Dropout (64-32): [64, 32], activation=relu
  - Moyenne (128-64-32): [128, 64, 32], activation=relu
  - Moyenne avec Dropout (128-64-32): [128, 64, 32], activation=relu
  - Profonde (256-128-64-32): [256, 128, 64, 32], activation=relu
  - Mixed Activations (128-64): [128, 64], activation=tanh


### 2.1 Comparaison des architectures

Entraînement et comparaison de plusieurs architectures pour déterminer la meilleure configuration.

In [6]:
# Entraînement et comparaison des architectures
# Nous allons entraîner quelques architectures sélectionnées
selected_architectures = architectures[:3]  # Sélectionner les 3 premières pour démonstration
results_comparison = []

# Réduire les données pour accélérer le test
# À adapter selon votre disponibilité en ressources
train_ratio = 0.3  # Utiliser 30% des données pour un test rapide
train_size = int(len(X_train) * train_ratio)
X_train_sample = X_train[:train_size]
y_train_sample = y_train[:train_size]

print(f"Entraînement sur {train_size} samples ({train_ratio*100:.0f}% des données)")
print("\n" + "="*80)

for arch_idx, architecture in enumerate(selected_architectures):
    for loss_name in loss_functions_to_test:
        try:
            print(f"\n[{arch_idx+1}/{len(selected_architectures)}] Entraînement: {architecture['name']} + {loss_name.upper()}")
            
            # Construire le réseau
            network, description = build_network(architecture, loss_name, X_train.shape[1])
            
            # Ajouter Early Stopping
            network.add_callback(EarlyStopping(patience=5))
            
            # Entraîner le réseau
            network.fit(
                X_train_sample, 
                y_train_sample,
                epochs=20,  # Nombre d'epochs réduit pour la démo
                batch_size=32,
                validation_split=0.2,
                learning_rate=0.01
            )
            
            # Évaluation sur les données de validation
            evaluator = Evaluation(X_validation, y_validation)
            metrics = evaluator.validate(network)
            
            result = {
                'architecture': description,
                'accuracy': metrics['Accuracy'].values[0],
                'precision': metrics['Precision'].values[0],
                'recall': metrics['Recall'].values[0],
                'f1_score': metrics['F1 Score'].values[0],
                'auc': metrics['AUC'].values[0],
                'network': network
            }
            results_comparison.append(result)
            
            print(f"  ✓ Accuracy: {result['accuracy']:.4f}, F1: {result['f1_score']:.4f}, AUC: {result['auc']:.4f}")
            
        except Exception as e:
            print(f"  ✗ Erreur: {str(e)}")

print("\n" + "="*80)
print("Comparaison terminée!")

# Convertir les résultats en DataFrame pour une meilleure visualisation
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'network'} for r in results_comparison])
print("\nRésultats de comparaison:")
print(results_df.to_string(index=False))

Entraînement sur 17874 samples (30% des données)


[1/3] Entraînement: Simple (64-32) + BCE
  ✗ Erreur: name 'build_network' is not defined

[1/3] Entraînement: Simple (64-32) + MSE
  ✗ Erreur: name 'build_network' is not defined

[2/3] Entraînement: Simple avec Dropout (64-32) + BCE
  ✗ Erreur: name 'build_network' is not defined

[2/3] Entraînement: Simple avec Dropout (64-32) + MSE
  ✗ Erreur: name 'build_network' is not defined

[3/3] Entraînement: Moyenne (128-64-32) + BCE
  ✗ Erreur: name 'build_network' is not defined

[3/3] Entraînement: Moyenne (128-64-32) + MSE
  ✗ Erreur: name 'build_network' is not defined

Comparaison terminée!

Résultats de comparaison:
Empty DataFrame
Columns: []
Index: []


In [7]:
# Sélection du meilleur modèle basé sur F1-score
best_result = max(results_comparison, key=lambda x: x['f1_score'])
best_architecture_name = best_result['architecture']
print(f"\n🏆 Meilleur modèle: {best_architecture_name}")
print(f"   Accuracy: {best_result['accuracy']:.4f}")
print(f"   F1-Score: {best_result['f1_score']:.4f}")
print(f"   AUC: {best_result['auc']:.4f}")

ValueError: max() iterable argument is empty

## 3. Construction et Entraînement du Réseau Final

### Utilisation de la meilleure architecture avec Early Stopping et callbacks

In [ ]:
# Extraction de la meilleure architecture à partir du résultat de comparaison
best_model_result = best_result
best_model = best_model_result['network']

print("Réentraînement du meilleur modèle sur l'ensemble des données d'entraînement...")
print(f"Architecture: {best_architecture_name}")
print(f"Données d'entraînement: {X_train.shape}")

# Ajouter les callbacks
best_model.add_callback(DrawRealTimeLoss(verbose=False))
best_model.add_callback(EarlyStopping(patience=10, verbose=True))

# Entraîner le modèle final
print("\nEntraînement en cours...")
best_model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    learning_rate=0.01,
    verbose=1
)

print("\n✓ Entraînement terminé!")

# Récupérer l'historique d'entraînement
if best_model.history:
    training_history = best_model.history
    print(f"Nombre d'epochs complétés: {len(training_history)}")
    print(f"Loss final: {training_history[-1]['loss']:.6f}")
    if 'val_loss' in training_history[-1]:
        print(f"Validation Loss final: {training_history[-1]['val_loss']:.6f}")

## 4. Entraînement, Évaluation et Métriques

### Calcul des métriques de performance

In [ ]:
# Évaluation complète du modèle
print("="*80)
print("ÉVALUATION DU MODÈLE")
print("="*80)

# Créer l'évaluateur
evaluator = Evaluation(X_validation, y_validation)
metrics_df = evaluator.validate(best_model)

print("\n📊 Métriques de performance sur les données de validation:")
print(metrics_df.to_string(index=False))

# Récupérer les composantes de la matrice de confusion
tp, fp, tn, fn = evaluator._confusion_matrix
print(f"\n📈 Matrice de confusion:")
print(f"   TP (True Positives):  {tp}")
print(f"   FP (False Positives): {fp}")
print(f"   TN (True Negatives):  {tn}")
print(f"   FN (False Negatives): {fn}")

# Extraire les métriques individuelles
accuracy = metrics_df['Accuracy'].values[0]
precision = metrics_df['Precision'].values[0]
recall = metrics_df['Recall'].values[0]
f1 = metrics_df['F1 Score'].values[0]
auc = metrics_df['AUC'].values[0]

print(f"\n✅ Résumé des métriques:")
print(f"   Accuracy:  {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")
print(f"   AUC:       {auc:.4f}")

In [ ]:
# Visualisation des courbes d'entraînement (Learning Curves)
print("\nGénération des graphiques...")

if best_model.history and len(best_model.history) > 0:
    # Extraire les valeurs de loss
    losses = [h.get('loss', np.nan) for h in best_model.history]
    val_losses = [h.get('val_loss', np.nan) for h in best_model.history]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Graphique 1: Loss vs Validation Loss
    epochs_range = range(1, len(losses) + 1)
    axes[0].plot(epochs_range, losses, label='Training Loss', marker='o')
    axes[0].plot(epochs_range, val_losses, label='Validation Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Courbes d\'apprentissage (Loss)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Graphique 2: Métriques de validation
    metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']
    metrics_values = [accuracy, precision, recall, f1, auc]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    axes[1].bar(metrics_names, metrics_values, color=colors, alpha=0.7)
    axes[1].set_ylabel('Score')
    axes[1].set_title('Métriques de Performance')
    axes[1].set_ylim([0, 1])
    for i, v in enumerate(metrics_values):
        axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Graphiques affichés")
else:
    print("Pas d'historique d'entraînement disponible")

In [ ]:
# Visualisation de la matrice de confusion et ROC curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
confusion_matrix_values = np.array([[tn, fp], [fn, tp]])
sns.heatmap(confusion_matrix_values, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'],
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Matrice de Confusion')
axes[0].set_ylabel('Valeur Réelle')
axes[0].set_xlabel('Prédiction')

# ROC Curve
try:
    # Obtenir les prédictions
    y_pred_proba = best_model.predict(X_validation)
    
    # Calculer la ROC curve manuelle
    thresholds = np.linspace(0, 1, 100)
    tpr_list = []
    fpr_list = []
    
    for threshold in thresholds:
        y_pred_binary = (y_pred_proba.flatten() >= threshold).astype(int)
        tp_roc = np.sum((y_pred_binary == 1) & (y_validation == 1))
        fp_roc = np.sum((y_pred_binary == 1) & (y_validation == 0))
        tn_roc = np.sum((y_pred_binary == 0) & (y_validation == 0))
        fn_roc = np.sum((y_pred_binary == 0) & (y_validation == 1))
        
        tpr = tp_roc / (tp_roc + fn_roc) if (tp_roc + fn_roc) > 0 else 0
        fpr = fp_roc / (fp_roc + tn_roc) if (fp_roc + tn_roc) > 0 else 0
        
        tpr_list.append(tpr)
        fpr_list.append(fpr)
    
    axes[1].plot(fpr_list, tpr_list, 'b-', linewidth=2, label=f'ROC Curve (AUC={auc:.3f})')
    axes[1].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
except Exception as e:
    axes[1].text(0.5, 0.5, f'Erreur: {str(e)}', ha='center', va='center')
    axes[1].set_title('ROC Curve')

plt.tight_layout()
plt.show()

print("✓ Visualisations créées")

### 4.1 Détection du Surapprentissage (Overfitting)

Analyse des courbes d'entraînement pour détecter le surapprentissage

In [ ]:
# Analyse du surapprentissage
print("="*80)
print("ANALYSE DU SURAPPRENTISSAGE")
print("="*80)

if best_model.history and len(best_model.history) > 1:
    losses = np.array([h.get('loss', np.nan) for h in best_model.history])
    val_losses = np.array([h.get('val_loss', np.nan) for h in best_model.history])
    
    # Calculer la différence entre train et validation loss
    loss_diff = val_losses - losses
    
    final_train_loss = losses[-1]
    final_val_loss = val_losses[-1]
    avg_overfitting = np.mean(loss_diff)
    
    print(f"\n📊 Analyse des courbes d'apprentissage:")
    print(f"   Loss d'entraînement final: {final_train_loss:.6f}")
    print(f"   Loss de validation final : {final_val_loss:.6f}")
    print(f"   Écart moyen (Val - Train): {avg_overfitting:.6f}")
    
    if final_val_loss > final_train_loss * 1.2:
        print(f"\n⚠️  SURAPPRENTISSAGE DÉTECTÉ")
        print(f"   La loss de validation est {final_val_loss/final_train_loss:.2f}x plus grande que la loss d'entraînement")
        print(f"   Recommandations:")
        print(f"   - Ajouter plus de Dropout")
        print(f"   - Utiliser une L1/L2 regularization")
        print(f"   - Augmenter la quantité de données d'entraînement")
    else:
        print(f"\n✅ PAS DE SURAPPRENTISSAGE SIGNIFICATIF")
        print(f"   Le modèle généralise bien")
    
    # Graphique d'analyse du surapprentissage
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs_range = range(1, len(losses) + 1)
    axes[0].plot(epochs_range, losses, label='Training Loss', marker='o', linewidth=2)
    axes[0].plot(epochs_range, val_losses, label='Validation Loss', marker='s', linewidth=2)
    axes[0].fill_between(epochs_range, losses, val_losses, alpha=0.2, color='gray', label='Écart')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Détection du Surapprentissage')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Graphique de l'écart
    axes[1].plot(epochs_range, loss_diff, marker='o', color='red', linewidth=2)
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[1].fill_between(epochs_range, loss_diff, 0, where=(loss_diff >= 0), alpha=0.3, color='red', label='Surapprentissage')
    axes[1].fill_between(epochs_range, loss_diff, 0, where=(loss_diff < 0), alpha=0.3, color='green', label='Sous-apprentissage')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Écart (Val Loss - Train Loss)')
    axes[1].set_title('Écart entre Validation et Training Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Historique d'entraînement insuffisant pour l'analyse du surapprentissage")

## 5. Analyse du Seuil de Décision

Variation du seuil de décision pour trouver la meilleure configuration

In [ ]:
print("="*80)
print("ANALYSE DU SEUIL DE DÉCISION")
print("="*80)

# Obtenir les prédictions en probabilités
try:
    y_pred_proba = best_model.predict(X_validation).flatten()
    
    # Tester différents seuils
    thresholds = np.linspace(0, 1, 21)
    threshold_results = []
    
    for threshold in thresholds:
        # Appliquer le seuil
        y_pred_binary = (y_pred_proba >= threshold).astype(int)
        
        # Calculer les métriques
        tp = np.sum((y_pred_binary == 1) & (y_validation == 1))
        fp = np.sum((y_pred_binary == 1) & (y_validation == 0))
        tn = np.sum((y_pred_binary == 0) & (y_validation == 0))
        fn = np.sum((y_pred_binary == 0) & (y_validation == 1))
        
        # Calculer les métriques
        accuracy_t = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0
        precision_t = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall_t = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_t = 2 * (precision_t * recall_t) / (precision_t + recall_t) if (precision_t + recall_t) > 0 else 0
        
        threshold_results.append({
            'threshold': threshold,
            'accuracy': accuracy_t,
            'precision': precision_t,
            'recall': recall_t,
            'f1_score': f1_t,
            'tp': tp,
            'fp': fp,
            'tn': tn,
            'fn': fn
        })
    
    # Créer un DataFrame des résultats
    threshold_df = pd.DataFrame(threshold_results)
    
    # Afficher les résultats
    print("\n📊 Résultats pour différents seuils:")
    print(threshold_df[['threshold', 'accuracy', 'precision', 'recall', 'f1_score']].round(4).to_string(index=False))
    
    # Trouver le meilleur seuil selon différentes métriques
    best_threshold_f1 = threshold_df.loc[threshold_df['f1_score'].idxmax(), 'threshold']
    best_threshold_accuracy = threshold_df.loc[threshold_df['accuracy'].idxmax(), 'threshold']
    best_threshold_recall = threshold_df.loc[threshold_df['recall'].idxmax(), 'threshold']
    best_threshold_precision = threshold_df.loc[threshold_df['precision'].idxmax(), 'threshold']
    
    print(f"\n🎯 Meilleurs seuils selon différentes métriques:")
    print(f"   F1-Score:  {best_threshold_f1:.2f} (F1={threshold_df.loc[threshold_df['threshold']==best_threshold_f1, 'f1_score'].values[0]:.4f})")
    print(f"   Accuracy:  {best_threshold_accuracy:.2f} (Acc={threshold_df.loc[threshold_df['threshold']==best_threshold_accuracy, 'accuracy'].values[0]:.4f})")
    print(f"   Recall:    {best_threshold_recall:.2f} (Rec={threshold_df.loc[threshold_df['threshold']==best_threshold_recall, 'recall'].values[0]:.4f})")
    print(f"   Precision: {best_threshold_precision:.2f} (Prec={threshold_df.loc[threshold_df['threshold']==best_threshold_precision, 'precision'].values[0]:.4f})")
    
    # Visualisation des seuils
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Graphique 1: Accuracy vs Threshold
    axes[0, 0].plot(threshold_df['threshold'], threshold_df['accuracy'], 'b-', marker='o', linewidth=2)
    axes[0, 0].axvline(x=best_threshold_accuracy, color='r', linestyle='--', label=f'Best: {best_threshold_accuracy:.2f}')
    axes[0, 0].set_xlabel('Decision Threshold')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Accuracy vs Threshold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Graphique 2: Precision, Recall, F1 vs Threshold
    axes[0, 1].plot(threshold_df['threshold'], threshold_df['precision'], 'r-', marker='o', label='Precision', linewidth=2)
    axes[0, 1].plot(threshold_df['threshold'], threshold_df['recall'], 'g-', marker='s', label='Recall', linewidth=2)
    axes[0, 1].plot(threshold_df['threshold'], threshold_df['f1_score'], 'b-', marker='^', label='F1-Score', linewidth=2)
    axes[0, 1].axvline(x=best_threshold_f1, color='black', linestyle='--', alpha=0.5)
    axes[0, 1].set_xlabel('Decision Threshold')
    axes[0, 1].set_ylabel('Score')
    axes[0, 1].set_title('Precision, Recall, F1 vs Threshold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Graphique 3: TP, FP, TN, FN vs Threshold
    axes[1, 0].plot(threshold_df['threshold'], threshold_df['tp'], 'g-', marker='o', label='TP', linewidth=2)
    axes[1, 0].plot(threshold_df['threshold'], threshold_df['fp'], 'r-', marker='o', label='FP', linewidth=2)
    axes[1, 0].plot(threshold_df['threshold'], threshold_df['tn'], 'b-', marker='s', label='TN', linewidth=2)
    axes[1, 0].plot(threshold_df['threshold'], threshold_df['fn'], 'orange', marker='s', label='FN', linewidth=2)
    axes[1, 0].set_xlabel('Decision Threshold')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Confusion Matrix Components vs Threshold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Graphique 4: Distribution des prédictions
    axes[1, 1].hist(y_pred_proba[y_validation == 0], bins=30, alpha=0.6, label='Classe 0', color='red')
    axes[1, 1].hist(y_pred_proba[y_validation == 1], bins=30, alpha=0.6, label='Classe 1', color='green')
    axes[1, 1].axvline(x=0.5, color='black', linestyle='--', label='Seuil par défaut (0.5)')
    axes[1, 1].axvline(x=best_threshold_f1, color='blue', linestyle='--', label=f'Meilleur seuil ({best_threshold_f1:.2f})')
    axes[1, 1].set_xlabel('Probabilité prédite')
    axes[1, 1].set_ylabel('Fréquence')
    axes[1, 1].set_title('Distribution des prédictions')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Graphiques de seuil créés")
    
except Exception as e:
    print(f"Erreur lors de l'analyse du seuil: {str(e)}")

## 6. Résumé et Conclusions

### Synthèse des résultats et recommandations

In [ ]:
print("="*80)
print("RÉSUMÉ ET CONCLUSIONS")
print("="*80)

print(f"\n📋 MODÈLE FINAL SÉLECTIONNÉ")
print(f"   Architecture: {best_architecture_name}")
print(f"   Epochs: {len(best_model.history) if best_model.history else 'N/A'}")

print(f"\n📊 PERFORMANCE GLOBALE")
print(f"   Accuracy:  {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")
print(f"   AUC:       {auc:.4f}")

print(f"\n🎯 MATRICE DE CONFUSION")
print(f"   TP: {tp} (vrais positifs)")
print(f"   FP: {fp} (faux positifs)")
print(f"   TN: {tn} (vrais négatifs)")
print(f"   FN: {fn} (faux négatifs)")

# Calculer le taux de faux positifs et faux négatifs
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\n📈 TAUX D'ERREUR")
print(f"   Faux Positive Rate: {fpr:.4f} ({fpr*100:.2f}%)")
print(f"   Faux Negative Rate: {fnr:.4f} ({fnr*100:.2f}%)")

print(f"\n💡 RECOMMANDATIONS")

# Analyser et donner des recommandations
if auc < 0.75:
    print(f"   ⚠️  AUC faible ({auc:.3f}): le modèle pourrait être amélioré")
    print(f"      - Augmenter la complexité du modèle")
    print(f"      - Ajouter plus de features")
    print(f"      - Augmenter le nombre d'epochs d'entraînement")
elif auc > 0.90:
    print(f"   ✅ AUC excellent ({auc:.3f}): le modèle performe très bien")
else:
    print(f"   ✓ AUC acceptable ({auc:.3f}): le modèle performe relativement bien")

if precision > recall:
    print(f"   - Precision > Recall: le modèle est plus conservateur dans ses prédictions")
    print(f"     (peu de faux positifs mais plus de faux négatifs)")
elif recall > precision:
    print(f"   - Recall > Precision: le modèle détecte plus de cas positifs")
    print(f"     (plus de faux positifs mais peu de faux négatifs)")

if fpr > 0.2:
    print(f"   - Faux Positive Rate élevé ({fpr*100:.1f}%): risque de fausses alarmes")
    
if fnr > 0.2:
    print(f"   - Faux Negative Rate élevé ({fnr*100:.1f}%): risque de manquer des vrais positifs")

print(f"\n✅ ANALYSE COMPLÉTÉE")
print("="*80)

### 6.1 Tableau comparatif - Résultats des architectures testées

In [ ]:
print("\n" + "="*80)
print("COMPARAISON DES ARCHITECTURES TESTÉES")
print("="*80)

if results_comparison:
    # Créer un DataFrame avec tous les résultats
    comparison_display = results_df.copy()
    
    # Ajouter un classement par F1-Score
    comparison_display['rank_f1'] = comparison_display['f1_score'].rank(ascending=False).astype(int)
    
    # Afficher les résultats triés par F1-Score
    print("\n📊 Classement par F1-Score:")
    print(comparison_display.sort_values('f1_score', ascending=False)[
        ['rank_f1', 'architecture', 'accuracy', 'precision', 'recall', 'f1_score', 'auc']
    ].to_string(index=False))
    
    # Visualisation comparative
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    architectures_short = [r['architecture'][:20] + '...' if len(r['architecture']) > 20 else r['architecture'] 
                          for r in results_comparison]
    
    # Graphique 1: Accuracy comparison
    axes[0, 0].bar(architectures_short, [r['accuracy'] for r in results_comparison], color='skyblue')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Accuracy par Architecture')
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Graphique 2: F1-Score comparison
    axes[0, 1].bar(architectures_short, [r['f1_score'] for r in results_comparison], color='lightgreen')
    axes[0, 1].set_ylabel('F1-Score')
    axes[0, 1].set_title('F1-Score par Architecture')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # Graphique 3: All metrics radar
    x_pos = np.arange(len(architectures_short))
    axes[1, 0].plot(x_pos, [r['accuracy'] for r in results_comparison], marker='o', label='Accuracy', linewidth=2)
    axes[1, 0].plot(x_pos, [r['precision'] for r in results_comparison], marker='s', label='Precision', linewidth=2)
    axes[1, 0].plot(x_pos, [r['recall'] for r in results_comparison], marker='^', label='Recall', linewidth=2)
    axes[1, 0].plot(x_pos, [r['f1_score'] for r in results_comparison], marker='d', label='F1-Score', linewidth=2)
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(architectures_short, rotation=45)
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_title('Toutes les Métriques par Architecture')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Graphique 4: AUC comparison
    axes[1, 1].bar(architectures_short, [r['auc'] for r in results_comparison], color='salmon')
    axes[1, 1].set_ylabel('AUC')
    axes[1, 1].set_title('AUC par Architecture')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualisations comparatives créées")
else:
    print("Aucun résultat de comparaison disponible")

# TEST

## 1. Chargement des données

Chargement des données d'entraînement et de validation à partir des fichiers CSV.

In [9]:
# Chargement des données
df_train = pd.read_csv('dataset/dataset_train.csv')
df_validation = pd.read_csv('dataset/dataset_validation.csv')

print(f"Données d'entraînement: {df_train.shape}")
print(f"Données de validation: {df_validation.shape}")
print(f"\nPremières lignes des données d'entraînement:")
print(df_train.head())
print(f"\nInformations sur les données:")
print(df_train.info())

Données d'entraînement: (59582, 17)
Données de validation: (22384, 17)

Premières lignes des données d'entraînement:
   Diabetes_binary  HighBP  HighChol  CholCheck      BMI  Smoker  Stroke  \
0              0.0     0.0       0.0        1.0  0.31250     0.0     0.0   
1              0.0     1.0       0.0        1.0  0.34375     1.0     0.0   
2              1.0     1.0       1.0        1.0  0.75000     0.0     0.0   
3              1.0     1.0       0.0        1.0  0.53125     0.0     0.0   
4              0.0     1.0       1.0        1.0  0.43750     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  \
0                   0.0           1.0     1.0      1.0                0.0   
1                   0.0           1.0     0.0      1.0                0.0   
2                   1.0           0.0     1.0      1.0                0.0   
3                   0.0           0.0     0.0      1.0                0.0   
4                   0.0           0.0    

In [10]:
# Séparation des features et de la cible
# La colonne cible est 'Diabetes_binary'
target_column = 'Diabetes_binary'

X_train = df_train.drop(columns=[target_column]).values.astype(np.float32)
y_train = df_train[target_column].values.astype(np.int32)

X_validation = df_validation.drop(columns=[target_column]).values.astype(np.float32)
y_validation = df_validation[target_column].values.astype(np.int32)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_validation shape: {X_validation.shape}, y_validation shape: {y_validation.shape}")
print(f"\nDistribution de y_train: {np.bincount(y_train)}")
print(f"Distribution de y_validation: {np.bincount(y_validation)}")

X_train shape: (59582, 16), y_train shape: (59582,)
X_validation shape: (22384, 16), y_validation shape: (22384,)

Distribution de y_train: [29791 29791]
Distribution de y_validation: [19074  3310]


## 2. Choix de l'Architecture du Réseau

### Variation de l'architecture avec différentes configurations
- Couches et nombre de neurones
- Fonctions d'activation : Sigmoid, ReLU, Tanh, Softmax
- Fonctions de perte : MSE, MAE, BCE, CCE
- Dropouts pour la régularisation

In [13]:
# Imports des fonctions d'activation et de perte disponibles
from lib.neural_network import NeuralNetwork, Layer, Evaluation
from lib.neural_network.activation.sigmoid import Sigmoid
from lib.neural_network.activation.relu import Relu
from lib.neural_network.activation.tanh import Tanh

from lib.neural_network.loss import (
    BinaryCrossEntropy,
    CategoricalCrossEntropy, 
    MeanSquaredError,
    MeanAbsoluteError
)

activations_dict = {
    'sigmoid': Sigmoid,
    'relu': Relu,
    'tanh': Tanh,
}

loss_dict = {
    'bce': BinaryCrossEntropy,
    'cce': CategoricalCrossEntropy,
    'mse': MeanSquaredError,
    'mae': MeanAbsoluteError
}

layers_list: list[Layer] = [
    Layer(16, 0.0, activation=activations_dict['relu']),
    Layer(8, 0.0, activation=activations_dict['relu']),
    Layer(1, 0.0, activation=activations_dict['sigmoid'])
] 

network = NeuralNetwork(layers = layers_list, loss = loss_dict['bce'](), inputs = X_train.shape[1])

print(X_train)

network.fit(
    X_train, 
    y_train, 
    epochs=32, 
    batch_size=32, 
    validation_split=0.2, 
    learning_rate=0.01
)

evaluator = Evaluation(X_validation, y_validation)
metrics = evaluator.validate(network)
print("\nMétriques de performance sur les données de validation:")
print(metrics.to_string(index=False))

[[0.        0.        1.        ... 0.        0.        0.75     ]
 [1.        0.        1.        ... 0.        0.        1.       ]
 [1.        1.        1.        ... 1.        0.        0.75     ]
 ...
 [1.        1.        1.        ... 0.        0.        0.75     ]
 [0.        1.        1.        ... 1.        0.        0.5833333]
 [0.        1.        1.        ... 0.        0.        0.8333333]]


TypeError: Relu.compute() missing 1 required positional argument: 'x'

In [17]:
x_train = np.random.rand(100, 5)
y_train = np.random.rand(100, 1)
x_val = np.random.rand(20, 5)
y_val = np.random.rand(20, 1)

GridSearch().search(
    {
        "learning_rate": [0.01, 0.001],
        "batch_size": [32, 64],
        "epochs": [10, 20],
        "loss": [MeanSquaredError()],
        "early_stopping_patience": [5],
        "early_stopping_delta": [0.001],
        "architecture": [
            [
                {"neurons": [10], "dropout_rate": [0.2], "activation": [Relu()]},
                {"neurons": [10], "dropout_rate": [0.2], "activation": [Sigmoid()]},
            ]
        ],
    },
    x_train,
    y_train,
    x_val,
    y_val,
)

ValueError: operands could not be broadcast together with shapes (32,10) (10,1) 